
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Exploring the Results of a DLT Pipeline



While DLT abstracts away many of the complexities associated with running production ETL on Databricks, many folks may wonder what's actually happening under the hood.

In this notebook, we'll avoid getting too far into the weeds, but will explore how data and metadata are persisted by DLT.

In [0]:
%run ./Includes/Classroom-Setup-04.3

## Querying Tables in the Target Database

As long as a target database is specified during DLT Pipeline configuration, tables should be available to users throughout your Databricks environment.

Run the cell below to see the tables registered to the database used in this demo.

In [0]:
%sql
USE ${DA.schema_name};

SHOW TABLES;

Note that the view we defined in our pipeline is absent from our tables list.

Query results from the **`orders_bronze`** table.

In [0]:
%sql
SELECT * FROM orders_bronze


Recall that **`orders_bronze`** was defined as a streaming table in DLT, but our results here are static.

Because DLT uses Delta Lake to store all tables, each time a query is executed, we will always return the most recent version of the table. But queries outside of DLT will return snapshot results from DLT tables, regardless of how they were defined.

## Examine Results of `APPLY CHANGES INTO`

Recall that the **customers_silver** table was implemented with changes from a CDC feed applied as Type 1 SCD.

Let's query this table below.

In [0]:
%sql
SELECT * FROM customers_silver


The **`customers_silver`** table correctly represents the current active state of our Type 1 table with changes applied. However, our **customers_silver** table is actually implemented as a view against a hidden table named **__apply_changes_storage_customers_silver**, which includes additional fields: **__Timestamp**, **__DeleteVersion**, and **__UpsertVersion**.

We can see this if we run **`DESCRIBE EXTENDED`**.

In [0]:
%sql
DESCRIBE EXTENDED customers_silver

If we query this hidden table, we'll see these 3 fields. However, users shouldn't need to interact directly with this table as it's just leveraged by DLT to ensure that updates are applied in the correct order to materialize results correctly.

In [0]:
%sql
SELECT * FROM __apply_changes_storage_customers_silver

## Examining Data Files

Run the following cell to look at the files in the configured **Storage location**.

In [0]:
files = dbutils.fs.ls(DA.paths.storage_location)
display(files)

The **autoloader** and **checkpoint** directories contain data used to manage incremental data processing with Structured Streaming.

The **system** directory captures events associated with the pipeline.

In [0]:
files = dbutils.fs.ls(f"{DA.paths.storage_location}/system/events")
display(files)

These event logs are stored as a Delta table. Let's query the table.

In [0]:
display(spark.sql(f"SELECT * FROM delta.`{DA.paths.storage_location}/system/events`"))

We'll dive deeper into the metrics in the notebook that follows.

Let's view the contents of the **tables** directory.

In [0]:
files = dbutils.fs.ls(f"{DA.paths.storage_location}/tables")
display(files)

Each of these directories contains a Delta Lake table being managed by DLT.


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>